# nano-prover SFT on Colab (free T4 GPU)

The Colab kernel runs on a **remote Google GPU machine** (files live under `/content`, not your laptop).
The `sft_train.py` that actually runs is the one **cloned by cell 2** — so whenever the code was updated,
**re-run cell 2** (it force-pulls the latest) before cell 5. **Prereq:** commit + push first. Run top to bottom.

In [13]:
# 1. confirm we have a GPU (must show Tesla T4 / similar; 'command not found' => CPU runtime, redo as GPU)
!nvidia-smi

Sat Oct 24 03:36:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
# 2. clone + FORCE to the latest main (re-run this whenever the code changed).  No nesting.
#    Private repo: replace the URL with https://<GITHUB_TOKEN>@github.com/FELIXGAO0315/Conjecture-Gym.git
import os
os.chdir('/content')
if not os.path.isdir('/content/Conjecture-Gym'):
    !git clone --depth 1 https://github.com/FELIXGAO0315/Conjecture-Gym.git
os.chdir('/content/Conjecture-Gym')
!git fetch -q --depth 1 origin main && git reset -q --hard origin/main
assert os.path.exists('output/train_corpus.jsonl'), 'corpus not found — did you push it?'
print('cwd:', os.getcwd(), '| HEAD:', end=' ')
!git log --oneline -1

cwd: /content/Conjecture-Gym | HEAD: 2003dff (grafted, HEAD -> main, origin/main, origin/HEAD) colab notebook: force-latest clone in cell 2, drop redundant torchao install, fix stale cell-5 comment


In [15]:
# 3. install the SFT stack.  (torchao is handled inside sft_train.py — it removes Colab's old one.)
!pip install -q -U transformers trl peft datasets accelerate

In [16]:
# 4. build train/test splits from the corpus
!python training/prepare_data.py

corpus 5642 → train 5360  test 282  → /content/Conjecture-Gym/training/data/


In [17]:
# 5. LoRA SFT — defaults: 1 epoch, max_seq 1024, grad-ckpt off  => ~335 steps, ~30 min on T4.
#    Faster first pass (~10 min): !SFT_MODEL=Qwen/Qwen2.5-Coder-0.5B-Instruct python training/sft_train.py
#    OOM? !SFT_GRAD_CKPT=1 python training/sft_train.py   |   more epochs? SFT_EPOCHS=3
!python training/sft_train.py

device: cuda:Tesla T4  dtype=torch.float16
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:02<00:00, 154.21it/s]
Generating train split: 5360 examples [00:00, 84992.89 examples/s]
Map: 100% 5360/5360 [00:00<00:00, 9969.95 examples/s] 
train examples: 5360
  (SFTConfig: this TRL version ignores: ['max_seq_length', 'warmup_ratio'] )
Adding EOS to train dataset: 100% 5360/5360 [00:00<00:00, 30935.45 examples/s]
Tokenizing train dataset: 100% 5360/5360 [00:09<00:00, 554.85 examples/s]
Building labels for train dataset: 100% 5360/5360 [00:02<00:00, 2400.55 examples/s]
Truncating train dataset: 100% 5360/5360 [00:01<00:00, 4128.50 examples/s]
Dropping fully masked examples from train dataset: 100% 5360/5360 [00:00<00:00, 17038.67 examples/s]
{'loss': '0.5323', 'grad_norm': '0.3548', 'learning_rate': '0.0001984', 'entropy': '0.5312', 'mean_token_accuracy': '0.8692', 'num_tokens': '2.346e+05', 'epoch': '0.0597'}
{'loss': '0.1248', 'grad_norm':

## Get the adapter back (`training/out/`), then evaluate LOCALLY
Pick one:

In [18]:
# OPTION A — zip for manual download (browser Colab: a download prompt; VS Code: find out.zip in the file tree)
!cd training && zip -rq out.zip out && ls -lh out.zip
try:
    from google.colab import files; files.download('training/out.zip')
except Exception as e:
    print('download via the file panel instead:', e)

-rw-r--r-- 1 root root 264M Oct 24 04:24 out.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
# OPTION B — push the adapter back to GitHub (needs a token with repo write).  Then `git pull` locally.
# TOKEN = 'ghp_xxx'   # a GitHub Personal Access Token (classic, repo scope)
# !git config user.email you@example.com && git config user.name you
# !git add -f training/out && git commit -m 'trained adapter'
# !git push https://{TOKEN}@github.com/FELIXGAO0315/Conjecture-Gym.git main

Back on your **local** machine (the Lean verifier lives there):
```bash
git pull                 # if you used Option B
# or unzip the downloaded out.zip into training/out/
python training/eval_lean.py --n 50 --k 1   # real Lean-verified pass@k
```